In [34]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset
import pandas as pd
import os
from glob import glob
from scipy.signal import decimate

In [35]:
def get_dataset_name(filename_with_dir):
    filename_without_dir = filename_with_dir.split("/")[-1]
    dataset_name = ".".join(filename_without_dir.split(".")[:-1])
    return dataset_name

filename_path = r"Final Project data\Intra\train\rest_105923_1.h5"

with h5py.File(filename_path, "r") as f:
    print("Keys:", list(f.keys()))

    dataset_name = get_dataset_name(filename_path)

    if dataset_name in f:
        matrix = f[dataset_name][()]
    else:
        # Use first dataset if names don't match
        first_key = list(f.keys())[0]
        matrix = f[first_key][()]

    print(type(matrix))
    print(matrix.shape)

print(matrix.shape)

Keys: ['rest_105923']
<class 'numpy.ndarray'>
(248, 35624)
(248, 35624)


In [36]:
def load_h5(filepath):
    with h5py.File(filepath, "r") as f:
        dataset_name = get_dataset_name(filepath)

        if dataset_name in f:
            data = f[dataset_name][()]
        else:
            first_key = list(f.keys())[0]
            data = f[first_key][()]

    return data

In [37]:
def extract_label(filepath):

    filename = os.path.basename(filepath).lower()

    if "rest" in filename:
        return 0

    elif "story" in filename or "math" in filename:
        return 1

    elif "memory" in filename:
        return 2

    elif "motor" in filename:
        return 3

    else:
        raise ValueError("Unknown class")

In [38]:
def create_windows(data, window_size=512, stride=256):
    windows = []

    for start in range(0, data.shape[1] - window_size, stride):
        end = start + window_size
        windows.append(data[:, start:end])

    return np.array(windows)

# z-score norm

In [39]:
def normalize(data):

    mean = np.mean(data, axis=1, keepdims=True)
    std = np.std(data, axis=1, keepdims=True)

    return (data - mean) / (std + 1e-8)

def downsample(data, factor=8):

    return decimate(data, factor, axis=1)

In [41]:
X = []
y = []

files = glob("Final Project data/Intra/train/*.h5")

for filepath in files:
    print(f"Processing {filepath}...")

    data = load_h5(filepath)

    data = normalize(data)

    data = downsample(data)

    windows = create_windows(data)

    label = extract_label(filepath)

    for w in windows:

        X.append(w)
        y.append(label)

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

Processing Final Project data/Intra/train\rest_105923_1.h5...
Processing Final Project data/Intra/train\rest_105923_2.h5...
Processing Final Project data/Intra/train\rest_105923_3.h5...
Processing Final Project data/Intra/train\rest_105923_4.h5...
Processing Final Project data/Intra/train\rest_105923_5.h5...
Processing Final Project data/Intra/train\rest_105923_6.h5...
Processing Final Project data/Intra/train\rest_105923_7.h5...
Processing Final Project data/Intra/train\rest_105923_8.h5...
Processing Final Project data/Intra/train\task_motor_105923_1.h5...
Processing Final Project data/Intra/train\task_motor_105923_2.h5...
Processing Final Project data/Intra/train\task_motor_105923_3.h5...
Processing Final Project data/Intra/train\task_motor_105923_4.h5...
Processing Final Project data/Intra/train\task_motor_105923_5.h5...
Processing Final Project data/Intra/train\task_motor_105923_6.h5...
Processing Final Project data/Intra/train\task_motor_105923_7.h5...
Processing Final Project dat